
# 01 - Expressive power: testing the theory instead of only proving it

**No training. Runs in a few minutes on CPU.** Produces Table 2 of the paper.

A positional encoding paper that proves an expressivity theorem and never tests it leaves
the reader unable to tell a real separation from a vacuous one. This notebook turns each
proposition into an executable check on an explicit witness graph pair.

We compare three colour refinements. Writing $c^t(v)$ for the colour of $v$ after $t$ rounds:

| | refinement rule |
|---|---|
| **1-WL** | $c^{t+1}(v)=\mathrm{HASH}\big(c^t(v),\ \{\!\{c^t(u):u\in N(v)\}\!\}\big)$ |
| **SPD-WL** | $c^{t+1}(v)=\mathrm{HASH}\big(c^t(v),\ \{\!\{(d(u,v),c^t(u)):u\neq v\}\!\}\big)$ |
| **SPcount-WL** | as SPD-WL, but the pair label is $(d(u,v),\ \#\mathrm{sp}(u,v))$ |

SPD-WL is the refinement our M3 encoding implements (at $r\geq\mathrm{diam}$); SPcount-WL is
what M4 adds. The propositions under test:

* **Prop. 2** - SPD-WL is *strictly* stronger than 1-WL. Needs a pair 1-WL cannot separate
  and SPD-WL can.
* **Prop. 3** - SPD-WL does *not* reach 3-WL. Needs a pair that 3-WL separates and ours
  cannot. This is an upper bound on our own method, and it is reported as such.
* **Prop. 4** - shortest-path counts strictly add power over hop distance alone. Needs a
  pair SPD-WL cannot separate and SPcount-WL can - found here by exhaustive search, together
  with a proof that no smaller witness exists.

**One implementation detail that matters.** Colours must be hashed by *content*, not
re-indexed per graph. If each graph maps its own signatures to `0,1,2,...` independently,
the colours are not comparable across graphs and every pair looks identical - a silent bug
that makes the whole notebook report "no separation" for everything.

In [ ]:
!pip -q install torch-geometric

In [ ]:

import os, sys, math, time, hashlib, itertools
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    RESULT_ROOT, DATA_ROOT = "/content/drive/MyDrive/trpe", "/content/trpe_data"
except Exception:
    RESULT_ROOT, DATA_ROOT = os.path.abspath("./trpe"), os.path.abspath("./trpe_data")

OUT_DIR = os.path.join(RESULT_ROOT, "results", "expressivity")
FIG_DIR = os.path.join(RESULT_ROOT, "figures")
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
print("out ->", OUT_DIR)


## The three refinements

In [ ]:

def all_pairs(A):
    """Full distance matrix and shortest-path counts, via (A^k)>0 first at k=d(u,v)."""
    n = A.shape[0]
    dist = np.full((n, n), -1, dtype=np.int64); np.fill_diagonal(dist, 0)
    nsp = np.zeros((n, n)); Mk = np.eye(n)
    for k in range(1, n):
        Mk = Mk @ A
        fresh = (Mk > 0.5) & (dist < 0)
        if not fresh.any(): break
        dist[fresh] = k; nsp[fresh] = Mk[fresh]
    return dist, nsp


def _h(x):
    return hashlib.md5(repr(x).encode()).hexdigest()[:16]


def refine(n, neighbours, max_rounds=8):
    """Generic colour refinement. `neighbours[v]` is a list of (pair_label, u).

    Colours are CONTENT hashes, never per-graph indices - otherwise two graphs would
    relabel their colours independently and could never be told apart.
    Returns the graph-level invariant: the multiset of stable node colours.
    """
    colour = ["0"] * n
    for _ in range(max_rounds):
        nxt = [_h((colour[v], tuple(sorted((repr(e), colour[u]) for e, u in neighbours[v]))))
               for v in range(n)]
        if nxt == colour:
            break
        colour = nxt
    return tuple(sorted(Counter(colour).items()))


def wl1(A):
    n = A.shape[0]
    return refine(n, [[(0, u) for u in np.nonzero(A[v])[0]] for v in range(n)])


def spd_wl(A, r=None):
    """What our M3 encoding implements: pairs labelled by hop distance."""
    n = A.shape[0]; d, _ = all_pairs(A); r = r or n
    lab = lambda x: int(x) if 1 <= x <= r else "inf"
    return refine(n, [[(lab(d[v, u]), u) for u in range(n) if u != v] for v in range(n)])


def spc_wl(A, r=None):
    """What M4 adds: pairs labelled by (hop distance, number of shortest paths)."""
    n = A.shape[0]; d, s = all_pairs(A); r = r or n
    lab = lambda x: int(x) if 1 <= x <= r else "inf"
    return refine(n, [[((lab(d[v, u]), float(s[v, u])), u) for u in range(n) if u != v]
                      for v in range(n)])


## Witness graphs

Each is built explicitly rather than downloaded, so the notebook has no data dependency.

* **Decalin vs bicyclopentyl** - the textbook 1-WL-indistinguishable pair: two fused
  six-rings against two five-rings joined by a bond. Same size, same degree sequence.
* **$4\times4$ rook's graph vs the Shrikhande graph** - both strongly regular with parameters
  $(16,6,2,2)$. Strongly regular graphs with equal parameters have *identical* distance
  distributions by definition, which is exactly why they bound our method from above. They
  are separated by 3-WL.
* **CSL$(41,k)$** - a 41-cycle plus skip links of length $k$. Every CSL graph is 4-regular
  and featureless, so 1-WL sees ten identical graphs and a plain GNN can only guess (10%).
  This is the dataset used by the `csl/` leaf notebooks.

In [ ]:

def from_edges(n, edges):
    A = np.zeros((n, n))
    for a, b in edges:
        A[a, b] = A[b, a] = 1.0
    return A


def csl(n=41, k=2):
    e = {tuple(sorted(t)) for t in
         [(i, (i + 1) % n) for i in range(n)] + [(i, (i + k) % n) for i in range(n)]}
    return from_edges(n, e)


def rook44():
    idx = {(i, j): i * 4 + j for i in range(4) for j in range(4)}
    return from_edges(16, [(idx[a], idx[b]) for a in idx for b in idx
                           if a < b and (a[0] == b[0] or a[1] == b[1])])


def shrikhande():
    """Cayley graph on Z4 x Z4 with connection set {+-(1,0), +-(0,1), +-(1,1)}."""
    S = [(1, 0), (3, 0), (0, 1), (0, 3), (1, 1), (3, 3)]
    idx = {(i, j): i * 4 + j for i in range(4) for j in range(4)}
    e = {tuple(sorted((idx[a], idx[((a[0] + s[0]) % 4, (a[1] + s[1]) % 4)]))) for a in idx for s in S}
    return from_edges(16, e)


decalin = from_edges(10, [(0,1),(1,2),(2,3),(3,4),(4,5),(5,0),(4,6),(6,7),(7,8),(8,9),(9,5)])
bicyclopentyl = from_edges(10, [(0,1),(1,2),(2,3),(3,4),(4,0),(5,6),(6,7),(7,8),(8,9),(9,5),(0,5)])

for nm, A in [("decalin", decalin), ("bicyclopentyl", bicyclopentyl),
              ("rook 4x4", rook44()), ("Shrikhande", shrikhande()),
              ("CSL(41,2)", csl(41, 2)), ("CSL(41,3)", csl(41, 3))]:
    deg = A.sum(1).astype(int)
    print(f"{nm:<15} n={A.shape[0]:>3}  m={int(A.sum()//2):>3}  degrees {deg.min()}-{deg.max()}")


## Table 2a - separation on the witness pairs

In [ ]:

PAIRS = [
    ("decalin / bicyclopentyl",        decalin,     bicyclopentyl, "Prop. 2"),
    ("CSL(41,2) / CSL(41,3)",          csl(41, 2),  csl(41, 3),    "Prop. 2"),
    ("CSL(41,4) / CSL(41,5)",          csl(41, 4),  csl(41, 5),    "Prop. 2"),
    ("4x4 rook / Shrikhande  SR(16,6,2,2)", rook44(), shrikhande(), "Prop. 3"),
]

rows = []
print(f"{'witness pair':<40}{'1-WL':>10}{'SPD-WL':>10}{'SPcount-WL':>13}   supports")
print("-" * 88)
for name, A, B, prop in PAIRS:
    r = {k: ("separates" if f(A) != f(B) else "-")
         for k, f in (("1-WL", wl1), ("SPD-WL", spd_wl), ("SPcount-WL", spc_wl))}
    rows.append(dict(pair=name, prop=prop, **r))
    print(f"{name:<40}{r['1-WL']:>10}{r['SPD-WL']:>10}{r['SPcount-WL']:>13}   {prop}")

print("""
Reading the table:
  Prop. 2 rows - 1-WL cannot separate these pairs but our encoding can, so the traversal
                 encoding is STRICTLY stronger than 1-WL. This is the lower bound.
  Prop. 3 row  - neither of our variants separates two strongly regular graphs with equal
                 parameters, because such graphs have identical distance distributions by
                 definition. 3-WL does separate them. This is our UPPER bound, and it is a
                 real limitation of the method rather than an artefact of the implementation.""")


## Table 2b - Proposition 4, by exhaustive search

Does counting shortest paths actually buy anything beyond the distance itself? Rather than
asserting it, we search every graph on $n$ nodes for a pair that SPD-WL cannot separate and
SPcount-WL can. Because SPcount-WL is an isomorphism invariant, any pair it separates is
automatically non-isomorphic - so no separate isomorphism check is needed.

Searching upward from $n=4$ also establishes **minimality**: the first $n$ at which a witness
appears is the smallest graph size on which the two refinements differ at all.

In [ ]:

def search_witness(n):
    pairs = list(itertools.combinations(range(n), 2))
    groups = defaultdict(list)
    for mask in range(1 << len(pairs)):
        edges = [pairs[i] for i in range(len(pairs)) if mask >> i & 1]
        if len(edges) < n - 1:
            continue
        A = from_edges(n, edges)
        if A.sum(1).min() == 0:            # skip graphs with isolated vertices
            continue
        groups[spd_wl(A)].append(edges)
    for members in groups.values():
        if len(members) < 2:
            continue
        by_spc = defaultdict(list)
        for e in members:
            by_spc[spc_wl(from_edges(n, e))].append(e)
        if len(by_spc) > 1:
            k = list(by_spc)
            return by_spc[k[0]][0], by_spc[k[1]][0], len(groups)
    return None, None, len(groups)


witness = None
for n in (4, 5, 6):
    t0 = time.time()
    g1, g2, ncls = search_witness(n)
    print(f"n={n}: {ncls:>4} SPD-WL classes, witness {'FOUND' if g1 else 'none'} "
          f"({time.time()-t0:.1f}s)")
    if g1:
        witness = (n, g1, g2)
        break

n, g1, g2 = witness
A, B = from_edges(n, g1), from_edges(n, g2)
print(f"\nMinimal witness on {n} nodes (none exists on fewer):")
for nm, M, E in (("G1", A, g1), ("G2", B, g2)):
    d, s = all_pairs(M); m = d >= 1
    print(f"  {nm}: E={E}")
    print(f"      degrees            {sorted(M.sum(1).astype(int))}")
    print(f"      distance multiset  {sorted(Counter(d[m].tolist()).items())}")
    print(f"      (d, #sp) multiset  {sorted(Counter(zip(d[m].tolist(), s[m].astype(int).tolist())).items())}")

print(f"\n  1-WL       : {'separates' if wl1(A)    != wl1(B)    else 'cannot separate'}")
print(f"  SPD-WL     : {'separates' if spd_wl(A) != spd_wl(B) else 'cannot separate'}")
print(f"  SPcount-WL : {'separates' if spc_wl(A) != spc_wl(B) else 'cannot separate'}")
print("""
The two graphs agree on size, degree sequence AND the complete multiset of pairwise
distances, and differ only in how many shortest paths realise those distances. Hop distance
alone therefore cannot separate them, which is exactly Proposition 4 - and it is what the
extra channel in condition M4 is for.""")


## Table 2c - how often does this matter on real data?

The witness pairs above are adversarial by construction. A fairer question is how much extra
separation the encoding buys on the actual benchmarks. For each dataset we take graphs of
equal size, find the pairs that 1-WL assigns the same signature, and measure the fraction of
those that our refinement separates.

This is a purely combinatorial measurement - no training, no learned parameters - so it
isolates representational capacity from optimisation.

In [ ]:

from torch_geometric.datasets import GNNBenchmarkDataset

def separation_rate(name, split=None, sample=120, r=None):
    ds = (GNNBenchmarkDataset(os.path.join(DATA_ROOT, name), name, split=split) if split
          else GNNBenchmarkDataset(os.path.join(DATA_ROOT, name), name))
    idx = np.linspace(0, len(ds) - 1, min(sample, len(ds))).astype(int)

    graphs = []
    for i in idx:
        d = ds[int(i)]
        n = int(d.num_nodes); ei = d.edge_index.numpy()
        A = np.zeros((n, n)); A[ei[0], ei[1]] = 1.0
        A = np.maximum(A, A.T); np.fill_diagonal(A, 0)
        graphs.append(A)

    # Only same-size graphs can collide under any of these refinements.
    by_size = defaultdict(list)
    for A in graphs:
        by_size[(A.shape[0], int(A.sum() // 2))].append(A)

    collide = sep_spd = sep_spc = 0
    for bucket in by_size.values():
        if len(bucket) < 2:
            continue
        sig1 = [wl1(A) for A in bucket]
        for i, j in itertools.combinations(range(len(bucket)), 2):
            if sig1[i] != sig1[j]:
                continue                    # 1-WL already separates them
            collide += 1
            sep_spd += spd_wl(bucket[i], r) != spd_wl(bucket[j], r)
            sep_spc += spc_wl(bucket[i], r) != spc_wl(bucket[j], r)
    return dict(dataset=name, sampled=len(graphs), wl_collisions=collide,
                spd_separated=sep_spd, spc_separated=sep_spc)


rows_c = []
for name, split in [("CSL", None), ("CLUSTER", "val"), ("CIFAR10", "val")]:
    t0 = time.time()
    row = separation_rate(name, split)
    rows_c.append(row)
    c = row["wl_collisions"]
    pct = lambda k: f"{100*row[k]/c:.1f}%" if c else "n/a"
    print(f"{name:<9} sampled {row['sampled']:>4} | 1-WL-indistinguishable pairs {c:>6} | "
          f"SPD-WL separates {pct('spd_separated'):>7} | SPcount-WL {pct('spc_separated'):>7} "
          f"({time.time()-t0:.0f}s)")

print("""
A dataset with few 1-WL collisions is one where expressivity is simply not the bottleneck -
that is a finding, not a failure, and it is the right lens for interpreting the accuracy
tables later. CSL is the opposite extreme: every graph collides under 1-WL by construction.""")

In [ ]:

import json
summary = {"witness_pairs": rows, "prop4_witness": {"n": n, "G1": g1, "G2": g2},
           "separation_rates": rows_c}
path = os.path.join(OUT_DIR, "expressivity.json")
json.dump(summary, open(path, "w"), indent=2, default=str)
print("written ->", path)
print("\nThis file feeds Table 2 in shared/07_results_and_figures.ipynb.")


## What this notebook establishes

| Proposition | Claim | Evidence here |
|---|---|---|
| **Prop. 2** | traversal RPE $\succ$ 1-WL, strictly | decalin/bicyclopentyl and CSL pairs separated by SPD-WL, not by 1-WL |
| **Prop. 3** | traversal RPE $\not\succeq$ 3-WL | $4\times4$ rook vs Shrikhande separated by neither variant |
| **Prop. 4** | shortest-path counts add power | minimal 6-node witness, found by exhaustive search; no witness on 5 nodes |

Together these place the method as $1\text{-WL}\subsetneq\text{T-RPE}\subseteq\text{GD-WL}\subsetneq 3\text{-WL}$.
Proposition 3 is a limitation of our own method and belongs in the paper as one - a method
whose upper bound is never stated invites the reader to assume there isn't one.